### Perio detection data set ###

In [31]:
import sys
import os
import cv2
from PIL import Image
import glob
import logging
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from pathlib import Path

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

%load_ext autoreload
%autoreload 2
import computervision
from computervision.imageproc import ImageData

# Print version info
print(f'Package version: {computervision.__version__}')
print(f'Python version:  {sys.version}')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Package version: v0.0.2
Python version:  3.12.3 (main, Jun 18 2025, 17:59:45) [GCC 13.3.0]


In [37]:
# Image directory
data_dir = os.environ.get('DATA')
image_dir = os.path.join(data_dir, 'dataset_object_260101')

# Output directory
#output_dir_name = 'enhanced_for_sam'
output_dir_name = 'original_rgb'
output_dir = os.path.join(image_dir, output_dir_name)
Path(output_dir).mkdir(exist_ok=True, parents=True)

file_list = glob.glob(os.path.join(image_dir, '*.png'))
print(len(file_list))

2139


In [13]:
def enhance_for_sam(
    rgb_image: np.ndarray,
    clip_limit: float = 3.0,
    tile_grid_size: tuple = (8, 8),
    alpha: float = 1.1,
    beta: float = 10
) -> np.ndarray:
    """
    Enhance contrast and brightness of an RGB image for better segmentation with SAM.

    Parameters
    ----------
    rgb_image : np.ndarray
        Input image in RGB format, dtype uint8, shape (H, W, 3).
    clip_limit : float, optional
        CLAHE clip limit; higher values give more contrast, default 3.0.
    tile_grid_size : tuple, optional
        CLAHE tile size; smaller tiles -> more local contrast.
    alpha : float, optional
        Global contrast gain (1.0 = no change). Used in linear transform: out = alpha * x + beta.
    beta : float, optional
        Global brightness offset (0 = no change). Same linear transform.

    Returns
    -------
    enhanced : np.ndarray
        Enhanced RGB image, dtype uint8, shape (H, W, 3).
    """

    if rgb_image.dtype != np.uint8:
        raise ValueError("rgb_image must be uint8 in range [0, 255].")

    # Convert RGB -> BGR for OpenCV, then to LAB
    bgr = cv2.cvtColor(rgb_image, cv2.COLOR_RGB2BGR)
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)

    # Split LAB channels
    L, A, B = cv2.split(lab)

    # Apply CLAHE on the L channel (adaptive contrast enhancement)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    L_clahe = clahe.apply(L)

    # Merge back
    lab_clahe = cv2.merge((L_clahe, A, B))

    # Convert back to BGR
    bgr_clahe = cv2.cvtColor(lab_clahe, cv2.COLOR_LAB2BGR)

    # Optional global linear brightness/contrast adjustment
    # new_image = alpha * image + beta
    enhanced_bgr = cv2.convertScaleAbs(bgr_clahe, alpha=alpha, beta=beta)

    # Convert back to RGB for SAM
    enhanced_rgb = cv2.cvtColor(enhanced_bgr, cv2.COLOR_BGR2RGB)

    return enhanced_rgb

In [30]:
file_new

'/app/data_model/dataset_object_260101/enhanced_for_sam/c59e44d7_20230410_14_enh.png'

In [38]:
f = 10
file = file_list[f]

for f, file in enumerate(file_list[:200]):

    im = ImageData().load_image(file)
    
    # Convert to RGB
    im_rgb = ImageData().np2color(im)
    
    # Enhance contrast
    im_eq = ImageData().hist_eq(im)
    
    # Use SAM function above
    im_sam = enhance_for_sam(im_eq)

    # Save the enhanced image
    file_name = os.path.splitext(os.path.basename(file))[0]
    file_name_new = f'{file_name}_rgb.png'
    file_new = os.path.join(output_dir, file_name_new)
    img_pil = Image.fromarray(im_rgb)
    img_pil.save(file_new)
    
    #fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(8, 4))
    #ax[0].imshow(im, cmap='grey')
    #ax[0].set_title('Original')
    #ax[1].imshow(im_sam)
    #ax[1].set_title('Enhanced')
    
    #for a in range(len(ax)):
    #    ax[a].set(xticks=[], yticks=[])
    
    #plt.show()

In [20]:
print(im_sam.shape)

(2400, 1708, 3)
